## Introduction
In this lab, you will explore a type of attack called *padding oracle attack*. Let's begin by breaking down what padding oracle attack means word by word.

### Padding Oracle Attack
**Padding**: There are encryption schemes (e.g., a block cipher in the CBC mode) that only support inputs of a fixed length or a multiple of a fixed length. This means it is impossible to use the encryption scheme to encrypt a message $m$ of an arbitrary length directly. Instead, an invertible padding function `pad()` is used so that the padded message $m \mathbin\Vert pad(m)$ has the target fixed length or a multiple of the fixed length.

**Oracle**: In a padding oracle attack. The oracle is typically a remote server that knows the decryption key. The attacker will ask the server to decrypt specially crafted messages, and the server will try to decrypt the messages and tell the attacker if the decryption succeeds or not. For encryption schemes with padding, a part of the decryption process is to check whether the padding is valid. If the server leaks whether the padding is valid to the attacker explicitly, then we say that the server is a *padding oracle*. (This is in contrast to sending the attacker a generic "decryption failure" message regardless of the type of failure that occured during decryption.)

**Attack**: The attacker uses the padding oracle to break the confidentiality of the encryption scheme. Typically, a padding oracle attack aims to recover the plaintext of a given ciphertext.

### Your Task
In this programming assignment, your goal is to implement a padding oracle attack against AES-CBC with a specific padding function. The attack allows an attacker to recover the plaintext of the ciphertext.

### Overview
The assignment is broken down into three components. 
In the first component, your goal is to implement AES-CBC with the said padding function. You are given an implementation of AES and you only need to implement the CBC mode and the padding function.
In the second component, you need to implement the core of the padding oracle attack: an attack that recovers one byte of the plaintext.
In the last component, you will extend the attack in the second component to recover the whole plaintext.

The first component is worth 20 points, the second component is worth 50 points, and the last component is worth 30 points. The whole assignment is worth 100 points. Usual rules regarding acedemic integrity apply.

## Part 1: AES-CBC and the Padding Function (20 points)
### Part 1.1: Padding Function (5 points)
Given a message $m$, we pad the length of $m$ to a multiple of 16 bytes. If $|m|$ is already a multiple of 16, we append a full block. The padding function is defined as follows:

- Let $l$ be the required padding length (in bytes; e.g., if the message is 18-byte long, $l$ should be 14).
- The padding function generates $l$ bytes where each byte equals to $l$ (e.g., a two-byte padding looks like (0x02, 0x02)). 
- The padding is appended to the message.

Implement the padding function as described above.

In [ ]:
def pad(m):
    l = (16-len(m) % 16)
    return m+(bytes([l]) * l)

### Part 1.2: AES-CBC (15 points)
Based on the AES implementation below (credit to [bluekeybo](https://github.com/bluekeybo)), implement AES-CBC. The encryption function of AES-CBC should take a message of an arbitrary length as an input. The padding should be done within the encryption function of AES-CBC.
You should also check the validity of the padding in the decryption function. If the validity check fails, you should return an error instead of outputting the decrypted value.

In [ ]:
s_box = (
    0x63, 0x7C, 0x77, 0x7B, 0xF2, 0x6B, 0x6F, 0xC5, 0x30, 0x01, 0x67, 0x2B, 0xFE, 0xD7, 0xAB, 0x76,
    0xCA, 0x82, 0xC9, 0x7D, 0xFA, 0x59, 0x47, 0xF0, 0xAD, 0xD4, 0xA2, 0xAF, 0x9C, 0xA4, 0x72, 0xC0,
    0xB7, 0xFD, 0x93, 0x26, 0x36, 0x3F, 0xF7, 0xCC, 0x34, 0xA5, 0xE5, 0xF1, 0x71, 0xD8, 0x31, 0x15,
    0x04, 0xC7, 0x23, 0xC3, 0x18, 0x96, 0x05, 0x9A, 0x07, 0x12, 0x80, 0xE2, 0xEB, 0x27, 0xB2, 0x75,
    0x09, 0x83, 0x2C, 0x1A, 0x1B, 0x6E, 0x5A, 0xA0, 0x52, 0x3B, 0xD6, 0xB3, 0x29, 0xE3, 0x2F, 0x84,
    0x53, 0xD1, 0x00, 0xED, 0x20, 0xFC, 0xB1, 0x5B, 0x6A, 0xCB, 0xBE, 0x39, 0x4A, 0x4C, 0x58, 0xCF,
    0xD0, 0xEF, 0xAA, 0xFB, 0x43, 0x4D, 0x33, 0x85, 0x45, 0xF9, 0x02, 0x7F, 0x50, 0x3C, 0x9F, 0xA8,
    0x51, 0xA3, 0x40, 0x8F, 0x92, 0x9D, 0x38, 0xF5, 0xBC, 0xB6, 0xDA, 0x21, 0x10, 0xFF, 0xF3, 0xD2,
    0xCD, 0x0C, 0x13, 0xEC, 0x5F, 0x97, 0x44, 0x17, 0xC4, 0xA7, 0x7E, 0x3D, 0x64, 0x5D, 0x19, 0x73,
    0x60, 0x81, 0x4F, 0xDC, 0x22, 0x2A, 0x90, 0x88, 0x46, 0xEE, 0xB8, 0x14, 0xDE, 0x5E, 0x0B, 0xDB,
    0xE0, 0x32, 0x3A, 0x0A, 0x49, 0x06, 0x24, 0x5C, 0xC2, 0xD3, 0xAC, 0x62, 0x91, 0x95, 0xE4, 0x79,
    0xE7, 0xC8, 0x37, 0x6D, 0x8D, 0xD5, 0x4E, 0xA9, 0x6C, 0x56, 0xF4, 0xEA, 0x65, 0x7A, 0xAE, 0x08,
    0xBA, 0x78, 0x25, 0x2E, 0x1C, 0xA6, 0xB4, 0xC6, 0xE8, 0xDD, 0x74, 0x1F, 0x4B, 0xBD, 0x8B, 0x8A,
    0x70, 0x3E, 0xB5, 0x66, 0x48, 0x03, 0xF6, 0x0E, 0x61, 0x35, 0x57, 0xB9, 0x86, 0xC1, 0x1D, 0x9E,
    0xE1, 0xF8, 0x98, 0x11, 0x69, 0xD9, 0x8E, 0x94, 0x9B, 0x1E, 0x87, 0xE9, 0xCE, 0x55, 0x28, 0xDF,
    0x8C, 0xA1, 0x89, 0x0D, 0xBF, 0xE6, 0x42, 0x68, 0x41, 0x99, 0x2D, 0x0F, 0xB0, 0x54, 0xBB, 0x16,
)

inv_s_box = (
    0x52, 0x09, 0x6A, 0xD5, 0x30, 0x36, 0xA5, 0x38, 0xBF, 0x40, 0xA3, 0x9E, 0x81, 0xF3, 0xD7, 0xFB,
    0x7C, 0xE3, 0x39, 0x82, 0x9B, 0x2F, 0xFF, 0x87, 0x34, 0x8E, 0x43, 0x44, 0xC4, 0xDE, 0xE9, 0xCB,
    0x54, 0x7B, 0x94, 0x32, 0xA6, 0xC2, 0x23, 0x3D, 0xEE, 0x4C, 0x95, 0x0B, 0x42, 0xFA, 0xC3, 0x4E,
    0x08, 0x2E, 0xA1, 0x66, 0x28, 0xD9, 0x24, 0xB2, 0x76, 0x5B, 0xA2, 0x49, 0x6D, 0x8B, 0xD1, 0x25,
    0x72, 0xF8, 0xF6, 0x64, 0x86, 0x68, 0x98, 0x16, 0xD4, 0xA4, 0x5C, 0xCC, 0x5D, 0x65, 0xB6, 0x92,
    0x6C, 0x70, 0x48, 0x50, 0xFD, 0xED, 0xB9, 0xDA, 0x5E, 0x15, 0x46, 0x57, 0xA7, 0x8D, 0x9D, 0x84,
    0x90, 0xD8, 0xAB, 0x00, 0x8C, 0xBC, 0xD3, 0x0A, 0xF7, 0xE4, 0x58, 0x05, 0xB8, 0xB3, 0x45, 0x06,
    0xD0, 0x2C, 0x1E, 0x8F, 0xCA, 0x3F, 0x0F, 0x02, 0xC1, 0xAF, 0xBD, 0x03, 0x01, 0x13, 0x8A, 0x6B,
    0x3A, 0x91, 0x11, 0x41, 0x4F, 0x67, 0xDC, 0xEA, 0x97, 0xF2, 0xCF, 0xCE, 0xF0, 0xB4, 0xE6, 0x73,
    0x96, 0xAC, 0x74, 0x22, 0xE7, 0xAD, 0x35, 0x85, 0xE2, 0xF9, 0x37, 0xE8, 0x1C, 0x75, 0xDF, 0x6E,
    0x47, 0xF1, 0x1A, 0x71, 0x1D, 0x29, 0xC5, 0x89, 0x6F, 0xB7, 0x62, 0x0E, 0xAA, 0x18, 0xBE, 0x1B,
    0xFC, 0x56, 0x3E, 0x4B, 0xC6, 0xD2, 0x79, 0x20, 0x9A, 0xDB, 0xC0, 0xFE, 0x78, 0xCD, 0x5A, 0xF4,
    0x1F, 0xDD, 0xA8, 0x33, 0x88, 0x07, 0xC7, 0x31, 0xB1, 0x12, 0x10, 0x59, 0x27, 0x80, 0xEC, 0x5F,
    0x60, 0x51, 0x7F, 0xA9, 0x19, 0xB5, 0x4A, 0x0D, 0x2D, 0xE5, 0x7A, 0x9F, 0x93, 0xC9, 0x9C, 0xEF,
    0xA0, 0xE0, 0x3B, 0x4D, 0xAE, 0x2A, 0xF5, 0xB0, 0xC8, 0xEB, 0xBB, 0x3C, 0x83, 0x53, 0x99, 0x61,
    0x17, 0x2B, 0x04, 0x7E, 0xBA, 0x77, 0xD6, 0x26, 0xE1, 0x69, 0x14, 0x63, 0x55, 0x21, 0x0C, 0x7D,
)


def sub_bytes(s):
    for i in range(4):
        for j in range(4):
            s[i][j] = s_box[s[i][j]]


def inv_sub_bytes(s):
    for i in range(4):
        for j in range(4):
            s[i][j] = inv_s_box[s[i][j]]


def shift_rows(s):
    s[0][1], s[1][1], s[2][1], s[3][1] = s[1][1], s[2][1], s[3][1], s[0][1]
    s[0][2], s[1][2], s[2][2], s[3][2] = s[2][2], s[3][2], s[0][2], s[1][2]
    s[0][3], s[1][3], s[2][3], s[3][3] = s[3][3], s[0][3], s[1][3], s[2][3]


def inv_shift_rows(s):
    s[0][1], s[1][1], s[2][1], s[3][1] = s[3][1], s[0][1], s[1][1], s[2][1]
    s[0][2], s[1][2], s[2][2], s[3][2] = s[2][2], s[3][2], s[0][2], s[1][2]
    s[0][3], s[1][3], s[2][3], s[3][3] = s[1][3], s[2][3], s[3][3], s[0][3]

def add_round_key(s, k):
    for i in range(4):
        for j in range(4):
            s[i][j] ^= k[i][j]


# learned from https://web.archive.org/web/20100626212235/http://cs.ucsb.edu/~koc/cs178/projects/JT/aes.c
xtime = lambda a: (((a << 1) ^ 0x1B) & 0xFF) if (a & 0x80) else (a << 1)


def mix_single_column(a):
    # see Sec 4.1.2 in The Design of Rijndael
    t = a[0] ^ a[1] ^ a[2] ^ a[3]
    u = a[0]
    a[0] ^= t ^ xtime(a[0] ^ a[1])
    a[1] ^= t ^ xtime(a[1] ^ a[2])
    a[2] ^= t ^ xtime(a[2] ^ a[3])
    a[3] ^= t ^ xtime(a[3] ^ u)


def mix_columns(s):
    for i in range(4):
        mix_single_column(s[i])


def inv_mix_columns(s):
    # see Sec 4.1.3 in The Design of Rijndael
    for i in range(4):
        u = xtime(xtime(s[i][0] ^ s[i][2]))
        v = xtime(xtime(s[i][1] ^ s[i][3]))
        s[i][0] ^= u
        s[i][1] ^= v
        s[i][2] ^= u
        s[i][3] ^= v

    mix_columns(s)


r_con = (
    0x00, 0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40,
    0x80, 0x1B, 0x36, 0x6C, 0xD8, 0xAB, 0x4D, 0x9A,
    0x2F, 0x5E, 0xBC, 0x63, 0xC6, 0x97, 0x35, 0x6A,
    0xD4, 0xB3, 0x7D, 0xFA, 0xEF, 0xC5, 0x91, 0x39,
)


def bytes2matrix(text):
    """ Converts a 16-byte array into a 4x4 matrix.  """
    return [list(text[i:i+4]) for i in range(0, len(text), 4)]

def matrix2bytes(matrix):
    """ Converts a 4x4 matrix into a 16-byte array.  """
    return bytes(sum(matrix, []))

def xor_bytes(a, b):
    """ Returns a new byte array with the elements xor'ed. """
    return bytes(i^j for i, j in zip(a, b))

def inc_bytes(a):
    """ Returns a new byte array with the value increment by 1 """
    out = list(a)
    for i in reversed(range(len(out))):
        if out[i] == 0xFF:
            out[i] = 0
        else:
            out[i] += 1
            break
    return bytes(out)


def split_blocks(message, block_size=16, require_padding=True):
        assert len(message) % block_size == 0 or not require_padding
        return [message[i:i+16] for i in range(0, len(message), block_size)]


class AES:
    """
    Class for AES-128 encryption with CBC mode and PKCS#7.

    This is a raw implementation of AES, without key stretching or IV
    management. Unless you need that, please use `encrypt` and `decrypt`.
    """
    rounds_by_key_size = {16: 10, 24: 12, 32: 14}
    def __init__(self, master_key):
        """
        Initializes the object with a given key.
        """
        assert len(master_key) in AES.rounds_by_key_size
        self.n_rounds = AES.rounds_by_key_size[len(master_key)]
        self._key_matrices = self._expand_key(master_key)

    def _expand_key(self, master_key):
        """
        Expands and returns a list of key matrices for the given master_key.
        """
        # Initialize round keys with raw key material.
        key_columns = bytes2matrix(master_key)
        iteration_size = len(master_key) // 4

        i = 1
        while len(key_columns) < (self.n_rounds + 1) * 4:
            # Copy previous word.
            word = list(key_columns[-1])

            # Perform schedule_core once every "row".
            if len(key_columns) % iteration_size == 0:
                # Circular shift.
                word.append(word.pop(0))
                # Map to S-BOX.
                word = [s_box[b] for b in word]
                # XOR with first byte of R-CON, since the others bytes of R-CON are 0.
                word[0] ^= r_con[i]
                i += 1
            elif len(master_key) == 32 and len(key_columns) % iteration_size == 4:
                # Run word through S-box in the fourth iteration when using a
                # 256-bit key.
                word = [s_box[b] for b in word]

            # XOR with equivalent word from previous iteration.
            word = xor_bytes(word, key_columns[-iteration_size])
            key_columns.append(word)

        # Group key words in 4x4 byte matrices.
        return [key_columns[4*i : 4*(i+1)] for i in range(len(key_columns) // 4)]

    def encrypt(self, plaintext):
        """
        Encrypts a single block of 16 byte long plaintext.
        """
        assert len(plaintext) == 16

        plain_state = bytes2matrix(plaintext)

        add_round_key(plain_state, self._key_matrices[0])

        for i in range(1, self.n_rounds):
            sub_bytes(plain_state)
            shift_rows(plain_state)
            mix_columns(plain_state)
            add_round_key(plain_state, self._key_matrices[i])

        sub_bytes(plain_state)
        shift_rows(plain_state)
        add_round_key(plain_state, self._key_matrices[-1])

        return matrix2bytes(plain_state)

    def decrypt(self, ciphertext):
        """
        Decrypts a single block of 16 byte long ciphertext.
        """
        assert len(ciphertext) == 16

        cipher_state = bytes2matrix(ciphertext)

        add_round_key(cipher_state, self._key_matrices[-1])
        inv_shift_rows(cipher_state)
        inv_sub_bytes(cipher_state)

        for i in range(self.n_rounds - 1, 0, -1):
            add_round_key(cipher_state, self._key_matrices[i])
            inv_mix_columns(cipher_state)
            inv_shift_rows(cipher_state)
            inv_sub_bytes(cipher_state)

        add_round_key(cipher_state, self._key_matrices[0])

        return matrix2bytes(cipher_state)

In [ ]:
def encrypt_cbc(plaintext, key, iv):
    aes = AES(key)
    plaintext = pad(plaintext) 
    ciphertext = b""
    previous_block = iv
    for i in range(0, len(plaintext), 16):
        block = plaintext[i:i+16]
        block = xor_bytes(block, previous_block)
        block = aes.encrypt(block)
        ciphertext += block
        previous_block = block

    return ciphertext


def decrypt_cbc(ciphertext, key, iv):
    aes = AES(key)
    plaintext = b""
    
    #Initiliaze the previouse block as the IV as the first "block"
    previous_block = iv
    for i in range(0, len(ciphertext), 16):
        block = ciphertext[i:i+16]
        decrypted_block = aes.decrypt(block)
        plaintext += xor_bytes(decrypted_block, previous_block)
        previous_block = block
    
    # Check padding validity
    if not check_pad(plaintext):
        return None
    
    return plaintext    
    

def check_pad(m):
    l = m[-1]  
    if l == 0 or l > 16:
        return False
    
    return all(b == l for b in m[-l:]) #Returns True if all bytes in the last l bytes of m are equal to l, otherwise returns False.


### Part 2: Recovering the Last Byte (50 points)
Without loss of generality, let $k$ be the decryption key and $(C_0, C_1)$ be the ciphertext ($C_0$ is the IV). Note that the decryption for the last block of the message (also the first block in this case) is $m_0 = Dec(k, C_1) \oplus C_0$. If the padding length is $l$ (in bytes), then the last $l$ bytes of $m_0$ should be $l$ each.
The unfortunate fact is that the attacker can manipulate $C_0$ and get a corresponding change in $m_0$. There are only two ciphertexts that will be decrypted without error (in the removal of the padding) if the attacker modifies the last byte of $m_0$ (by modifying the last byte of $C_0$), namely $m_0$ itself and $m_0$ except that the last byte is 0x01. This means the attacker can use an exhaustive search (255 attempts at most) to recover the last byte of the message.

Implement an attack that recovers the last byte of the message.

In [ ]:
def recover_last_byte(key, iv, ciphertext):
    
    C0 = iv
    C1 = ciphertext[:16]
    
    for guess in range(256):
        C0_modified = C0[:-1] + bytes([guess])
        

        if C0_modified == C0:
            continue
        
        result = decrypt_cbc(C1, key, C0_modified)
        
        if result is not None:
            last_byte = 0x01 ^ guess ^ C0[-1]
            return last_byte

### Part 3: Recovering the Whole Plaintext (30 points)
In fact, the attack in part 2 allows the attacker to remove all of the padding bytes. Extend your attack to recover the actual plaintext.

In [1]:
def recover_plaintext(key, iv, ciphertext):
    block_size = 16
    blocks = split_blocks(ciphertext, block_size)
    plaintext = b""

    previous_block = iv
    for C1 in blocks:
        intermediate = [0] * block_size 
        recovered_block = [0] * block_size

        for byte_index in reversed(range(block_size)):
            padding_val = block_size - byte_index

            for guess in range(256):
                modified_block = bytearray(previous_block)

                for k in range(byte_index + 1, block_size):
                    modified_block[k] = intermediate[k] ^ padding_val

                modified_block[byte_index] = guess

                result = decrypt_cbc(C1, key, bytes(modified_block))
                if result is not None:
                    intermediate[byte_index] = result[byte_index] ^ guess
                   
                    recovered_block[byte_index] = intermediate[byte_index] ^ previous_block[byte_index]
                    break

        plaintext += bytes(recovered_block)
        previous_block = C1

    return plaintext

### Remarks
1. You have seen in the lectures that AES-CBC is a CPA-secure cipher. So why it is still possible for an attacker to break its confidentiality? What capability have we allowed the attacker that is not a part of the CPA security game? Try to formulate a new security game that captures the new capability. (We will see this formally in the lectures later.)
2. The attack you have implemented allows an attacker to recover the plaintext of a ciphertext, but it does not allow you to recover the decryption key. This is not a glitch in the security definition and it is very normal in attacks.
3. You may think that adding integrity protection (by using a MAC) is sufficient to prevent this attack. It turns out that this is not always the case. In particular, SSL 3.0 and TLS 1.0 (roughly speaking, TLS is a rebrand of SSL) pack a message $m$ as $m \mathbin\Vert MAC(K_{MAC}, m) \mathbin\Vert pad(m \mathbin\Vert MAC(K_{MAC}, m))$ before encryption. During decryption, the validity of the padding is checked *before* the validity of the MAC, and the "invalid padding" message is sent to the client explicitly. So nothing is stopping the attack you have implemented from working! For more padding oracle attacks on the CBC mode, see the following paper: [Serge Vaudenay. Security Flaws Induced by CBC Padding Applications to SSL, IPSEC, WTLS...](https://www.iacr.org/archive/eurocrypt2002/23320530/cbc02_e02d.pdf).
4. The reason why the key is referred to as the decryption key in this assignment is that there are also padding oracle attacks on public-key encryption schemes (where the encryption key and decryption key are different). You will see an example of a padding oracle attack on a public-key encryption scheme in a later assignment.